In [1]:
import os
import shutil
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
SOURCE_DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/HockeyVision/dataset/dataset"
SITUATION_DATASET_PATH = "/content/situation_dataset_ready"

In [5]:
print("Existuje cesta:", os.path.exists(SOURCE_DATASET_PATH))
print("Obsah zdrojového datasetu:", os.listdir(SOURCE_DATASET_PATH))

Existuje cesta: True
Obsah zdrojového datasetu: ['neni_hokej', 'klidova_situace', 'vhazovani', 'jizda_s_pukem', 'souboj_u_mantinelu', 'zakrok_brankare']


In [6]:
if os.path.exists(SITUATION_DATASET_PATH):
    shutil.rmtree(SITUATION_DATASET_PATH)

situation_folders = [
    "jizda_s_pukem",
    "klidova_situace",
    "souboj_u_mantinelu",
    "vhazovani",
    "zakrok_brankare"
]

for folder in situation_folders:
    src_folder = os.path.join(SOURCE_DATASET_PATH, folder)
    dst_folder = os.path.join(SITUATION_DATASET_PATH, folder)
    shutil.copytree(src_folder, dst_folder)

print("Situační dataset připraven.")
print("Obsah:", os.listdir(SITUATION_DATASET_PATH))

Situační dataset připraven.
Obsah: ['zakrok_brankare', 'jizda_s_pukem', 'klidova_situace', 'souboj_u_mantinelu', 'vhazovani']


In [7]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 12
DATASET_PATH = SITUATION_DATASET_PATH

In [8]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

print("Třídy:", train_ds.class_names)

Found 731 files belonging to 5 classes.
Using 585 files for training.
Found 731 files belonging to 5 classes.
Using 146 files for validation.
Třídy: ['jizda_s_pukem', 'klidova_situace', 'souboj_u_mantinelu', 'vhazovani', 'zakrok_brankare']


In [9]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [10]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

In [11]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [12]:
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(5, activation="softmax")(x)

model = keras.Model(inputs, outputs)

In [13]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [14]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        "hockey_situations_model.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/12
19/19 ━━━━━━━━━━━━━━━━━━━━ 176s 9s/step - accuracy: 0.4906 - loss: 1.4015 - val_accuracy: 0.6301 - val_loss: 1.0278
Epoch 2/12
19/19 ━━━━━━━━━━━━━━━━━━━━ 178s 9s/step - accuracy: 0.5863 - loss: 1.1378 - val_accuracy: 0.6781 - val_loss: 0.9020
Epoch 3/12
19/19 ━━━━━━━━━━━━━━━━━━━━ 175s 8s/step - accuracy: 0.6274 - loss: 0.9924 - val_accuracy: 0.6849 - val_loss: 0.8658
Epoch 4/12
19/19 ━━━━━━━━━━━━━━━━━━━━ 213s 9s/step - accuracy: 0.6821 - loss: 0.9183 - val_accuracy: 0.6644 - val_loss: 0.8503
Epoch 5/12
19/19 ━━━━━━━━━━━━━━━━━━━━ 197s 8s/step - accuracy: 0.7026 - loss: 0.8889 - val_accuracy: 0.6849 - val_loss: 0.8015
Epoch 6/12
19/19 ━━━━━━━━━━━━━━━━━━━━ 199s 8s/step - accuracy: 0.6957 - loss: 0.8406 - val_accuracy: 0.7192 - val_loss: 0.7756
Epoch 7/12
19/19 ━━━━━━━━━━━━━━━━━━━━ 197s 8s/step - accuracy: 0.7128 - loss: 0.8086 - val_accuracy: 0.7329 - val_loss: 0.7545
Epoch 8/12
19/19 ━━━━━━━━━━━━━━━━━━━━ 215s 8s/step - accuracy: 0.7419 - loss: 0.7441 - val_accuracy: 0.7123 - v

In [15]:
model.save("hockey_situations_model.keras")
print("Model uložen jako hockey_situations_model.keras")

Model uložen jako hockey_situations_model.keras


In [16]:
from google.colab import files
files.download("hockey_situations_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>